# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [14]:
# Load the libraries as required.
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, QuantileTransformer, OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, make_scorer

In [15]:
# Load the CSV file
# df = pd.read_csv('../../05_src/data/fires/forestfires.csv')
# Display the first few rows
# print(df.head())

# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [16]:
# 
X = fires_dt.drop(columns=['area'])  
print("Features shape:", X.shape)

Features shape: (517, 12)


In [17]:
y = fires_dt['area']          

# Check shapes and types

print("Target shape:", y.shape)
print("Feature columns:", X.columns.tolist())

Target shape: (517,)
Feature columns: ['coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [ ]:
# Categorical and numerical vars
categorical_cols = ['month', 'day']
numerical_cols = [col for col in X.columns if col not in categorical_cols]


# Preprocessor 1
preproc1 = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols), # Standard scaling for numerical, 
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols) # one-hot for categorical
])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [ ]:
# --- Preprocessor 2 ---

categorical_cols = [col for col in ['month', 'day'] if col in X.columns] # categorical 
numerical_cols = [col for col in X.columns if col not in categorical_cols] # numerical vars

# non-linear transformation 
nonlinear_transform_cols = [col for col in ['rain', 'wind', 'ISI'] if col in numerical_cols]
linear_scale_cols = [col for col in numerical_cols if col not in nonlinear_transform_cols]

# preproc2 
preproc2 = ColumnTransformer(transformers=[
    ('quantile', QuantileTransformer(output_distribution='normal'), nonlinear_transform_cols),
    ('scale_rest', StandardScaler(), linear_scale_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [20]:
# RMSE scoring and cross-validation
rmse_scorer = make_scorer(mean_squared_error, squared=False)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [32]:
# Pipeline A = preproc1 + baseline

pipe_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])

param_grid_A = {
    'regressor__alpha': [0.1, 1.0, 10.0]
}

grid_A = GridSearchCV(pipe_A, param_grid_A, scoring=rmse_scorer, cv=cv)
grid_A.fit(X, y)
print("Pipeline A - Best RMSE:", -grid_A.best_score_)
print("Pipeline A - Best Params:", grid_A.best_params_)


Pipeline A - Best RMSE: -55.6258984022365
Pipeline A - Best Params: {'regressor__alpha': 0.1}


In [33]:
print(X.columns.tolist())

['coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']


In [37]:
# Pipeline B = preproc2 + baseline
pipe_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', Ridge())
])

param_grid_B = {
    'regressor__alpha': [0.1, 1.0, 10.0]
}

grid_B = GridSearchCV(pipe_B, param_grid_B, scoring=rmse_scorer, cv=cv)
grid_B.fit(X, y)
print("Pipeline B - Best RMSE:", -grid_B.best_score_)
print("Pipeline B - Best Params:", grid_B.best_params_)

Pipeline B - Best RMSE: -55.51366612550275
Pipeline B - Best Params: {'regressor__alpha': 0.1}


c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-pack

In [24]:
# Pipeline C = preproc1 + advanced model
pipe_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid_C = {
    'regressor__n_estimators': [100],
    'regressor__max_depth': [5, 10, None]
}

grid_C = GridSearchCV(pipe_C, param_grid_C, scoring=rmse_scorer, cv=cv)
grid_C.fit(X, y)
print("Pipeline C - Best RMSE:", -grid_C.best_score_)
print("Pipeline C - Best Params:", grid_C.best_params_)

Pipeline C - Best RMSE: -65.84303591966469
Pipeline C - Best Params: {'regressor__max_depth': None, 'regressor__n_estimators': 100}


In [38]:
# Pipeline D = preproc2 + advanced model
pipe_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid_D = {
    'regressor__n_estimators': [100],
    'regressor__max_depth': [5, 10, None]
}

grid_D = GridSearchCV(pipe_D, param_grid_D, scoring=rmse_scorer, cv=cv)
grid_D.fit(X, y)
print("Pipeline D - Best RMSE:", -grid_D.best_score_)
print("Pipeline D - Best Params:", grid_D.best_params_)
    

c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-pack

Pipeline D - Best RMSE: -65.58686119121118
Pipeline D - Best Params: {'regressor__max_depth': None, 'regressor__n_estimators': 100}


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [41]:
# Pipeline A: preproc1 + Ridge
param_grid_A = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0]
}

grid_A = GridSearchCV(pipe_A, param_grid_A, scoring=rmse_scorer, cv=cv)
grid_A.fit(X, y)

print("Pipeline A - Best RMSE:", -grid_A.best_score_)
print("Pipeline A - Best Params:", grid_A.best_params_)

Pipeline A - Best RMSE: -55.67680145817222
Pipeline A - Best Params: {'regressor__alpha': 0.01}


In [42]:
#Pipeline B: preproc2 + Ridge
param_grid_B = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0]
}

grid_B = GridSearchCV(pipe_B, param_grid_B, scoring=rmse_scorer, cv=cv)
grid_B.fit(X, y)

print("Pipeline B - Best RMSE:", -grid_B.best_score_)
print("Pipeline B - Best Params:", grid_B.best_params_)

c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-pack

Pipeline B - Best RMSE: -55.576541492484274
Pipeline B - Best Params: {'regressor__alpha': 0.01}


c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (517). n_quantiles is set to n_samples.
  warnings.warn(


In [43]:
# Pipeline C: preproc1 + RandomForest
param_grid_C = {
    'regressor__max_depth': [5, 10],
    'regressor__n_estimators': [100, 200]
}

grid_C = GridSearchCV(pipe_C, param_grid_C, scoring=rmse_scorer, cv=cv)
grid_C.fit(X, y)

print("Pipeline C - Best RMSE:", -grid_C.best_score_)
print("Pipeline C - Best Params:", grid_C.best_params_)

Pipeline C - Best RMSE: -65.77513725581241
Pipeline C - Best Params: {'regressor__max_depth': 10, 'regressor__n_estimators': 100}


In [44]:
# Pipeline D: preproc2 + RandomForest
param_grid_D = {
    'regressor__max_depth': [5, 10],
    'regressor__n_estimators': [100, 200]
}

grid_D = GridSearchCV(pipe_D, param_grid_D, scoring=rmse_scorer, cv=cv)
grid_D.fit(X, y)

print("Pipeline D - Best RMSE:", -grid_D.best_score_)
print("Pipeline D - Best Params:", grid_D.best_params_)

c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (413). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_data.py:2627: UserWarning: n_quantiles (1000) is greater than the total number of samples (414). n_quantiles is set to n_samples.
  warnings.warn(
c:\Users\sanjb\miniconda3\envs\dsi_participant\lib\site-pack

Pipeline D - Best RMSE: -65.13293493133021
Pipeline D - Best Params: {'regressor__max_depth': 10, 'regressor__n_estimators': 100}


In [51]:
# Collect results
results = {
    'Pipeline': ['A', 'B', 'C', 'D'],
    'Description': [
        'preproc1 + Ridge',
        'preproc2 + Ridge',
        'preproc1 + RF',
        'preproc2 + RF'
    ],
    'RMSE': [
        -grid_A.best_score_,
        -grid_B.best_score_,
        -grid_C.best_score_,
        -grid_D.best_score_
    ],
    'Best Params': [
        grid_A.best_params_,
        grid_B.best_params_,
        grid_C.best_params_,
        grid_D.best_params_
    ]
}

df_results = pd.DataFrame(results)
print(df_results)

  Pipeline       Description       RMSE  \
0        A  preproc1 + Ridge -55.676801   
1        B  preproc2 + Ridge -55.576541   
2        C     preproc1 + RF -65.775137   
3        D     preproc2 + RF -65.132935   

                                         Best Params  
0                         {'regressor__alpha': 0.01}  
1                         {'regressor__alpha': 0.01}  
2  {'regressor__max_depth': 10, 'regressor__n_est...  
3  {'regressor__max_depth': 10, 'regressor__n_est...  


In [52]:
grid_C.cv_results_['std_test_score']

array([27.44351473, 28.0427504 , 26.9126692 , 27.56682622])

# Evaluate

+ Which model has the best performance?

-- Pipeline C with preproc1 and Random forest Model has the best performance as it has the lower root mean square error value of -65.77. 

# Export

+ Save the best performing model to a pickle file.

In [53]:
import pickle

# Save into pickle file
best_model = grid_C.best_estimator_

with open("best_model_pipeline_C.pkl", "wb") as f:
    pickle.dump(best_model, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [59]:
# pip install shap

In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
best_model.fit(X_train, y_train)

# Get preprocessor and trained regressor
preprocessor = best_model.named_steps['preprocessing']
model = best_model.named_steps['regressor']

# Transform the test features
X_test_transformed = preprocessor.transform(X_test)

# Get feature names (from one-hot and scaled features)
feature_names = best_model.named_steps['preprocessing'].get_feature_names_out()

In [ ]:
import shap

explainer = shap.Explainer(model, X_test_transformed, feature_names=feature_names, check_additivity=False)
shap_values = explainer(X_test_transformed)

### error: could not fix it.

*(Answer here.)*

- I would remove features that had very small or zero contribution from the SHAP bar plot. 

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.